# Local PostgreSQL Automated Installation

### ⚠️ Requirements:
1. Download the PostgreSQL **18.6 Windows x86-64** installer from [EnterpriseDB](https://www.enterprisedb.com/downloads/postgres-postgresql-downloads). Select version 18.6 and the Windows x86-64 installer (`postgresql-18.6-5-windows-x64.exe`).
2. The installer file is approximately **400 MB**. After downloading it, copy `postgresql-18.6-5-windows-x64.exe` into the same folder as this `install_postgres.ipynb` notebook before starting the installation.
3. **Important:** When running the installation cell, a Windows UAC prompt will ask for administrator privileges. Click **Yes** or **Confirm** to allow the installation on your machine.

### ℹ️ Installation Information:
* **Software Version:** PostgreSQL 18.6-5 (64-Bit for Windows)
* **Default Target Path:** `C:\Programme\PostgreSQL\18`

### 📁 How to Change the Installation Directory:
If you want to override where PostgreSQL is installed, modify the **`INSTALL_DIR`** variable in the code cell below:
```python
INSTALL_DIR = r"C:\Your\Custom\Folder\Path"
```

In [ ]:
import os
import ctypes
import sys
import subprocess

# 1. Define configuration settings
# CHANGE THIS PATH IF YOU WANT TO OVERRIDE THE INSTALLATION DIRECTORY:
INSTALL_DIR = r"C:\Programme\PostgreSQL\18"
DATA_DIR = os.path.join(INSTALL_DIR, "data")
ADMIN_USER = "devi-group3"
ADMIN_PASS = "fh-group3-admin"
EXE_NAME = "postgresql-18.6-5-windows-x64.exe"

execute_install = True
print("[+] Pre-flight check: Verifying system status before installation...")

# CHECK A: Is the installer already running right now?
try:
    tasks = subprocess.check_output('tasklist /NH /FI "IMAGENAME eq ' + EXE_NAME + '"', shell=True, text=True)
    if EXE_NAME in tasks:
        print(f"\n[!] MESSAGE: An installation process of '{EXE_NAME}' is already running in the background.")
        print("    Please wait for it to complete. A new installation will not be started.")
        execute_install = False
except Exception:
    pass

# CHECK B: Is Port 5432 already occupied (PostgreSQL already installed/running)?
if execute_install:
    try:
        # Use native Windows netstat to check if port 5432 is listening
        netstat = subprocess.check_output('netstat -ano', shell=True, text=True)
        if ":5432 " in netstat:
            print("\n==================================================")
            print("[!] MESSAGE: PostgreSQL appears to already be installed and active (Port 5432 is busy)!")
            print("    The installation process is skipped because the database is ready.")
            print("==================================================")
            execute_install = False
        else:
            print("[+] System check passed: No active service detected on Port 5432.")
    except Exception:
        print("[+] System check passed (Skipped netstat verification).")

# 2. Find the path of the installer file
if execute_install:
    script_dir = os.getcwd()
    installer_path = os.path.join(script_dir, EXE_NAME)
    print(f"Searching for installer at: {installer_path}")

    # 3. Verify installer presence
    if not os.path.exists(installer_path):
        print(f"[-] Error: '{EXE_NAME}' not found in the directory.")
        print("    Please place the .exe file in the same folder as this notebook and try again.")
    else:
        print("[+] Installer file found! Preparing arguments...")
        
        # 4. Construct parameters string for Windows Shell execution
        args = (
            f'--mode unattended '
            f'--unattendedmodeui minimal '
            f'--prefix "{INSTALL_DIR}" '
            f'--datadir "{DATA_DIR}" '
            f'--superaccount {ADMIN_USER} '
            f'--superpassword {ADMIN_PASS} '
            f'--servicepassword {ADMIN_PASS}'
        )
        
        print(f"[+] Configuration:")
        print(f"    Target Folder: {INSTALL_DIR}")
        print(f"    DB User:       {ADMIN_USER}")
        
        print("\n[+] Requesting local machine administrative rights...")
        print("--> A WINDOWS POP-UP CONFIRMATION WINDOW WILL APPEAR ON YOUR SCREEN NOW. CLICK 'YES' TO PROCEED! <--")
        
        # 5. Execute using 'runas' verb to enforce elevated privileges just for the installer
        result = ctypes.windll.shell32.ShellExecuteW(None, "runas", installer_path, args, None, 1)
        
        if result > 32:
            print("[+] Windows Administrator confirmation received or bypassed successfully!")
            print("    The installer is now setting up the background service. Wait until the progress bar disappears.")
            print("    Afterward, execute the verification cell below.")
        else:
            print(f"[-] Elevation failed or user denied the Windows admin pop-up confirmation prompt (Error Code: {result}).")
else:
    print("\n[.] Installation skipped (Database is already present or running).")

### Step 6: Post-Installation Status Verification (Using psql.exe)
Run the cell below after the installer window disappears. It calls the newly installed native `psql.exe` via command prompt to fetch the active database engine version details.

In [ ]:
import time

psql_path = os.path.join(INSTALL_DIR, "bin", "psql.exe")
print(f"[+] Looking for native CLI client at: {psql_path}")

if not os.path.exists(psql_path):
    print("[-] Verification blocked: psql.exe not found. The installation directory hasn't been populated yet.")
else:
    print("[+] CLI client found. Attempting to query DB engine version via subprocess...")
    
    # Set environment variables for authentication to avoid interactive prompt freezes
    env = os.environ.copy()
    env["PGPASSWORD"] = ADMIN_PASS
    
    # Command to run psql directly against the engine system registry
    cmd = [psql_path, "-h", "localhost", "-p", "5432", "-U", ADMIN_USER, "-d", "postgres", "-c", "SELECT version();"]
    
    connected = False
    for attempt in range(1, 13):
        try:
            result = subprocess.run(cmd, env=env, capture_output=True, text=True, check=True)
            print("\n==================================================")
            print("[🎉] SUCCESS: PostgreSQL is successfully installed and running!")
            print(f"[⚙️] Database Engine Details:\n{result.stdout.strip()}")
            print("==================================================")
            connected = True
            break
        except subprocess.CalledProcessError:
            print(f"[.] Attempt {attempt}/12: Database service initialization in progress, retrying in 5s...")
            time.sleep(5)

    if not connected:
        print("\n[-] Connection verification failed. The service might be starting slowly or is blocked by network rules.")

## Optional: Use the PostgreSQL Extension in VS Code

To run queries directly from VS Code instead of using the notebook's `psql.exe` interface, install the **PostgreSQL** extension by **Chris Kolkman**:

1. Open the Extensions view with `Ctrl+Shift+X`, search for **PostgreSQL**, and install the extension by Chris Kolkman.
2. Click the PostgreSQL/database icon in the Activity Bar and choose **Add Connection**.
3. Enter the connection details:
   - **Host:** `localhost`
   - **Port:** `5432`
   - **User:** `devi-group3`
   - **Password:** `fh-group3-admin`
   - **Database:** `housing`
4. Right-click the connection and select **New Query**.
5. Run a query such as:

   ```sql
   SELECT * FROM housing_table;
   ```

The extension displays query results in a sortable grid. This is optional; the existing `psql.exe` workflow continues to work without it.